# Calibration diagnostics

This notebook checks whether Cobasket's long-only evidence score behaves consistently before relying on calibrated probabilities. It tests sign conventions, per-basket behaviour, overlapping outcomes, and the continuous evidence/outcome relation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cobasket import calibrate_watchlist
from cobasket.evidence import fit_probability_calibration


## 1. Load calibration records

Use the Parquet file generated by `cobasket-calibrate --records-out calibration_records.parquet`.


In [ ]:
records_path = Path('../calibration_records.parquet')
records = pd.read_parquet(records_path)
records['evaluation_date'] = pd.to_datetime(records['evaluation_date'])
records['future_date'] = pd.to_datetime(records['future_date'])
print(f'Rows: {len(records):,}')
print(f'Unique evaluation dates: {records.evaluation_date.nunique():,}')
print(f'Baskets: {records.basket.nunique():,}')
records.groupby('basket').agg(records=('ticker', 'size'), evaluations=('evaluation_date', 'nunique'))


## 2. Sign convention

For a spread $s = \sum_i w_i p_i$, the mean-reversion direction for one ticker is proportional to `-z * w`. The Johansen vector is only defined up to an overall sign, so flipping both `z` and `w` must leave this quantity unchanged.


In [ ]:
records['sign_case'] = np.select(
    [
        (records.z_score >= 0) & (records.weight >= 0),
        (records.z_score >= 0) & (records.weight < 0),
        (records.z_score < 0) & (records.weight >= 0),
        (records.z_score < 0) & (records.weight < 0),
    ],
    ['z+, w+', 'z+, w-', 'z-, w+', 'z-, w-'],
    default='unclassified',
)
records.groupby('sign_case').agg(
    n=('ticker', 'size'),
    mean_score=('score', 'mean'),
    mean_excess_return=('excess_return', 'mean'),
    outperform_rate=('outperformed', 'mean'),
).sort_index()


In [ ]:
driver = -records['z_score'].to_numpy() * records['weight'].to_numpy()
flipped_driver = -(-records['z_score'].to_numpy()) * (-records['weight'].to_numpy())
print('Maximum absolute sign-flip difference:', np.max(np.abs(driver - flipped_driver)))
assert np.allclose(driver, flipped_driver)

sign_check = records.loc[:, ['score', 'z_score', 'weight']].copy()
sign_check['expected_direction'] = np.sign(-sign_check['z_score'] * sign_check['weight'])
sign_check['score_direction'] = np.sign(sign_check['score'])
sign_check['direction_agrees'] = sign_check['expected_direction'] == sign_check['score_direction']
sign_check['direction_agrees'].value_counts(dropna=False)


In [ ]:
records.loc[~sign_check['direction_agrees'], ['basket', 'evaluation_date', 'ticker', 'score', 'z_score', 'weight']].head(20)


## 3. Calibration by basket

Pooling assumes a score has broadly the same meaning across baskets. Compare both probabilities and sample counts.


In [ ]:
score_edges = (-1.0, -0.60, -0.25, 0.25, 0.60, 1.0)
per_basket_tables = {}
for basket, group in records.groupby('basket'):
    calibration = fit_probability_calibration(group, score_edges=score_edges, horizon=20)
    table = calibration.table.copy()
    table.insert(0, 'basket', basket)
    per_basket_tables[basket] = table
per_basket = pd.concat(per_basket_tables.values(), ignore_index=True)
per_basket


In [ ]:
pivot_probability = per_basket.pivot(index='basket', columns='score_lower', values='probability_mean')
pivot_count = per_basket.pivot(index='basket', columns='score_lower', values='sample_count')
display(pivot_probability)
display(pivot_count)


## 4. Evidence versus future excess return

The scatter plot and equal-count bins show whether the relationship is monotonic, asymmetric, or essentially flat.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(records['score'], records['excess_return'], alpha=0.2, s=15)
ax.axhline(0.0, linewidth=1, linestyle='--')
ax.axvline(0.0, linewidth=1, linestyle='--')
ax.set_xlabel('Evidence score')
ax.set_ylabel('Future excess return versus equal-weight basket')
ax.set_title('Historical calibration records')
plt.show()

quantile_bin = pd.qcut(records['score'], q=10, duplicates='drop')
continuous_summary = records.groupby(quantile_bin, observed=True).agg(
    score_mean=('score', 'mean'),
    mean_excess_return=('excess_return', 'mean'),
    median_excess_return=('excess_return', 'median'),
    outperform_rate=('outperformed', 'mean'),
    n=('ticker', 'size'),
).reset_index(drop=True)
continuous_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(continuous_summary['score_mean'], continuous_summary['outperform_rate'], marker='o')
ax.axhline(0.5, linewidth=1, linestyle='--')
ax.set_xlabel('Mean evidence score in quantile bin')
ax.set_ylabel('Observed outperformance fraction')
ax.set_ylim(0, 1)
ax.set_title('Empirical probability versus evidence score')
plt.show()


## 5. Overlapping versus non-overlapping outcomes

The original calibration used `horizon=20, step=5`, so neighbouring outcomes overlap. Repeating with `step=20` gives fewer but much less correlated historical evaluations.


In [ ]:
portfolio_path = Path('../portfolio.json')
nonoverlap = calibrate_watchlist(
    portfolio_path,
    train_window=252,
    horizon=20,
    step=20,
)
display(nonoverlap.basket_summary)
nonoverlap.calibration.table


In [ ]:
overlap_calibration = fit_probability_calibration(records, score_edges=score_edges, horizon=20).table.copy()
comparison = overlap_calibration[['score_lower', 'score_upper', 'sample_count', 'probability_mean']].rename(
    columns={'sample_count': 'n_step5', 'probability_mean': 'p_step5'}
)
comparison['n_step20'] = nonoverlap.calibration.table['sample_count'].to_numpy()
comparison['p_step20'] = nonoverlap.calibration.table['probability_mean'].to_numpy()
comparison


## 6. Per-basket evidence curves


In [ ]:
for basket, group in records.groupby('basket'):
    if group['score'].nunique() < 4:
        continue
    bins = pd.qcut(group['score'], q=min(5, group['score'].nunique()), duplicates='drop')
    summary = group.groupby(bins, observed=True).agg(
        score_mean=('score', 'mean'),
        outperform_rate=('outperformed', 'mean'),
        mean_excess_return=('excess_return', 'mean'),
        n=('ticker', 'size'),
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(summary['score_mean'], summary['outperform_rate'], marker='o')
    ax.axhline(0.5, linewidth=1, linestyle='--')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Evidence score')
    ax.set_ylabel('Observed outperformance fraction')
    ax.set_title(basket)
    plt.show()
    display(summary)


## 7. Interpretation

- If the sign test fails, fix the evidence implementation before interpreting calibration.
- If all baskets are flat or non-monotonic, the current score may not predict long-only relative outperformance.
- If only some baskets behave well, basket-specific calibration or basket-quality gating may be preferable.
- If `step=5` looks useful but `step=20` does not, overlapping outcomes may be inflating apparent evidence.
- If negative scores are informative but positive scores are not, the long-only relationship may genuinely be asymmetric.
